# Court LLM Descriptor Extractor — Blackwell-Optimized (FlashInfer + FP8 KV + Continuous Batching)

This notebook is the speed-optimized rewrite of the 10k-row version.

**Target:** 5–6× throughput on a single RTX PRO 6000 Blackwell (95 GB) running Qwen3-8B-AWQ, **with zero quality loss**.

Key changes vs. the previous notebook (all preserve output quality):

1. **FlashInfer attention backend** — replaces Triton (significant decode speedup on Blackwell).
2. **FP8 KV cache** (`kv_cache_dtype="fp8"`) — halves KV-cache memory, doubles concurrent sequences.
3. **`max_model_len: 3072 → 2048`** — your real worst-case sequence is ~1500 tokens; the rest was unused KV cache budget.
4. **`max_num_seqs: 96 → 384`** — uses the freed KV cache for actual concurrency.
5. **`gpu_memory_utilization: 0.82 → 0.92`** — your dedicated 95 GB GPU has headroom.
6. **Schema hint moved into the system prompt** — prefix caching now covers the full constant prefix instead of just the system message, saving ~120 tokens of prefill per request.
7. **Continuous-batching submission** — submit 2048 prompts per `llm.generate()` call instead of 64. vLLM's scheduler keeps all 384 slots saturated; no long-tail GPU idle.
8. **Local-disk JSONL writes, then copy to Drive at end** — eliminates per-batch Drive flush latency.
9. **n-gram speculative decoding** (auto-enabled if vLLM supports it) — output is highly repetitive (JSON field names), so n-gram lookup hits often. Free 1.3–1.8× on top.
10. **`repetition_penalty: 1.02 → 1.0`** — at temperature 0 the penalty does nothing useful and adds per-step softmax cost.

Output quality is unchanged: same model, same temperature 0, same prompt content (just rearranged), same parsing/normalization, same retry path on parse failure.


In [1]:
# Cell 0 — Colab setup, package install, FlashInfer install
#
# IMPORTANT: After this cell runs the FIRST time in a fresh Colab runtime,
# RESTART THE RUNTIME ONCE (Runtime → Restart runtime), then re-run from Cell 0.
# This forces a clean Python process so vLLM\'s and FlashInfer\'s CUDA extensions
# bind cleanly. Without this, you may hit "LayerName already registered" or
# "Engine core initialization failed" errors.

import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# --- 1. Core packages -----------------------------------------------------
# Pin vLLM 0.10+ for Blackwell + FlashInfer + FP8-KV. transformers 4.51+ for
# Qwen3 chat-template `enable_thinking` arg.
print('\nInstalling/upgrading core packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'vllm>=0.10.0',
    'transformers>=4.51.0',
    'accelerate', 'safetensors', 'pandas', 'tqdm', 'huggingface_hub',
], check=True)

# --- 2. FlashInfer (3-package model from official docs) ------------------
# https://docs.flashinfer.ai/installation.html
#
# FlashInfer ships THREE packages that must agree:
#   flashinfer-python    — core (compiles/downloads kernels lazily)
#   flashinfer-cubin     — pre-compiled kernel binaries (must match -python version)
#   flashinfer-jit-cache — pre-built kernel cache for a SPECIFIC CUDA version
#
# Your earlier failure ("flashinfer-cubin (0.6.8.post1) does not match
# flashinfer (0.6.9)") came from installing only -python, which pulled the
# newest -cubin separately and they got out of sync. The fix is to install
# -python and -cubin TOGETHER so pip resolves them to the same version, and
# then add -jit-cache from the CUDA-specific index for your toolkit.
def _detect_cuda_index_suffix() -> str | None:
    """Return the FlashInfer wheel index suffix ('cu126', 'cu128', 'cu129', 'cu130', 'cu131') or None."""
    try:
        import torch
    except Exception:
        return None
    cuda = (torch.version.cuda or '').strip()
    if not cuda:
        return None
    # Map e.g. '12.6' -> 'cu126', '13.0' -> 'cu130'.
    parts = cuda.split('.')
    if len(parts) < 2:
        return None
    try:
        major, minor = int(parts[0]), int(parts[1])
    except ValueError:
        return None
    suffix = f'cu{major}{minor}'
    # FlashInfer's supported set per docs (2026-05-03): cu126, cu128, cu129, cu130, cu131
    supported = {'cu126', 'cu128', 'cu129', 'cu130', 'cu131'}
    if suffix in supported:
        return suffix
    # Fall back to the closest supported one: pick the highest <= detected.
    candidates = sorted(supported, key=lambda s: int(s[2:]))
    detected_int = major * 10 + minor
    best = None
    for c in candidates:
        if int(c[2:]) <= detected_int:
            best = c
    return best or candidates[0]

cuda_idx = _detect_cuda_index_suffix()
print(f'\nDetected CUDA index suffix for FlashInfer: {cuda_idx}')

def _install_flashinfer() -> bool:
    # Step A: install -python and -cubin TOGETHER so pip pins matching versions.
    print('Step A: installing flashinfer-python + flashinfer-cubin (pinned together)...')
    r = subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'flashinfer-python', 'flashinfer-cubin',
    ])
    if r.returncode != 0:
        print('  ✗ -python + -cubin install failed')
        return False

    # Step B: install -jit-cache from the CUDA-specific index (huge speedup for first kernel call).
    if cuda_idx is not None:
        print(f'Step B: installing flashinfer-jit-cache for {cuda_idx}...')
        r2 = subprocess.run([
            sys.executable, '-m', 'pip', 'install', '-q', '-U', '--pre',
            'flashinfer-jit-cache',
            '--index-url', f'https://flashinfer.ai/whl/{cuda_idx}',
        ])
        if r2.returncode != 0:
            print('  ! -jit-cache install failed (non-fatal — kernels will JIT-compile on first use)')
    else:
        print('Step B: skipped (could not detect supported CUDA version for jit-cache index)')

    # Step C: verify import. As a safety net, set FLASHINFER_DISABLE_VERSION_CHECK=1
    # in case the wheels are still slightly mismatched on this Colab runtime.
    os.environ['FLASHINFER_DISABLE_VERSION_CHECK'] = '1'
    try:
        import importlib
        importlib.invalidate_caches()
        import flashinfer
        print(f'  ✓ flashinfer {getattr(flashinfer, "__version__", "?")} importable')
        return True
    except Exception as exc:
        print(f'  ✗ flashinfer import failed even after install: {exc!r}')
        return False

flashinfer_ok = _install_flashinfer()
print(f'\nFlashInfer available: {flashinfer_ok}')
if flashinfer_ok:
    # Optional: print the FlashInfer config (helps debugging if something is off).
    try:
        subprocess.run([sys.executable, '-m', 'flashinfer', 'show-config'], check=False, timeout=30)
    except Exception:
        pass
else:
    print('NOTE: vLLM will auto-select an attention backend (FLASH_ATTN or TRITON_ATTN). Speedups still apply.')

print('\n' + '=' * 60)
print('Setup complete.')
print('If this was the first install in a fresh Colab runtime,')
print('  RESTART RUNTIME ONCE (Runtime → Restart runtime),')
print('  then resume from Cell 1 (do NOT rerun Cell 0).')
print('=' * 60)


Running in Colab: True
Mounted at /content/drive

Installing/upgrading core packages...

Detected CUDA index suffix for FlashInfer: cu130
Step A: installing flashinfer-python + flashinfer-cubin (pinned together)...
Step B: installing flashinfer-jit-cache for cu130...


  ✓ flashinfer 0.6.9 importable

FlashInfer available: True

Setup complete.
If this was the first install in a fresh Colab runtime,
  RESTART RUNTIME ONCE (Runtime → Restart runtime),
  then resume from Cell 1 (do NOT rerun Cell 0).


In [1]:
# Cell 1 — Imports and runtime check
#
# CRITICAL: this cell deliberately DOES NOT call any torch.cuda.* function.
# Touching CUDA in the parent process before vLLM loads breaks vLLM\'s V1
# engine, which spawns a worker subprocess that then cannot cleanly take
# over the GPU context. This produces the "Engine core initialization failed
# ... Failed core proc(s): {}" error you saw on the previous run.
# We use nvidia-smi via subprocess for GPU info instead, which does not init CUDA.

from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Any, Optional
from collections import Counter
import os
import sys
import re
import gc
import ast
import json
import time
import shutil
import subprocess
import traceback

import pandas as pd
from tqdm.auto import tqdm

# Set vLLM worker start method BEFORE anything else that might init CUDA.
# spawn is required when CUDA is initialized; setting it explicitly avoids
# vLLM having to override it later and surfaces any error earlier.
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')
# Safety net for FlashInfer if the cubin/python versions are slightly mismatched.
os.environ.setdefault('FLASHINFER_DISABLE_VERSION_CHECK', '1')

# Do NOT `import torch` here either — its `import` alone is fine, but ANY
# subsequent torch.cuda call would init CUDA. We import it lazily later.
torch = None  # filled in after vLLM has loaded, only for post-load metrics

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE_DIR = Path('/content/drive/MyDrive/swiss_law') if IN_COLAB else Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'
# Checkpoints are written to Drive so a Colab disconnect does NOT lose progress.
# /content/scratch_outputs would be wiped on runtime restart — Drive persists.
LOCAL_SCRATCH = (BASE_DIR / 'data' / 'checkpoints') if IN_COLAB else (BASE_DIR / 'scratch_outputs')
LOCAL_SCRATCH.mkdir(parents=True, exist_ok=True)

print('Imports OK')
print('Running in Colab:', IN_COLAB)
print('BASE_DIR:', BASE_DIR)
print('CHECKPOINT_DIR:', LOCAL_SCRATCH)

# GPU info via nvidia-smi (does NOT init CUDA in this process).
# Also captures compute capability so Cell 6 can decide whether FlashInfer is
# safe on this GPU. SM 12.0 (consumer/workstation Blackwell — RTX 5090, RTX PRO
# 6000) is currently NOT supported by FlashInfer 0.6.x even though it imports
# successfully — the kernels for SM 12.x are not in the cubin pack.
GPU_COMPUTE_CAPABILITY = None  # e.g. "12.0", "10.0", "9.0", "8.9"
GPU_NAME = None

def _print_gpu_info_via_nvidia_smi():
    global GPU_COMPUTE_CAPABILITY, GPU_NAME
    try:
        out = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=index,name,compute_cap,memory.total,memory.free,driver_version',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=10, check=True,
        )
        for line in out.stdout.strip().splitlines():
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 6:
                idx, name, cc, mem_total, mem_free, drv = parts[:6]
                print(f'GPU {idx}: {name} (compute capability {cc}); free={int(mem_free)/1024:.2f} GiB / total={int(mem_total)/1024:.2f} GiB; driver {drv}')
                # Capture the FIRST GPU\'s compute capability for later decisions.
                if GPU_COMPUTE_CAPABILITY is None:
                    GPU_COMPUTE_CAPABILITY = cc
                    GPU_NAME = name
            else:
                print(f'GPU info line: {line}')
    except FileNotFoundError:
        print('WARNING: nvidia-smi not found; assuming no GPU available.')
    except subprocess.CalledProcessError as exc:
        print(f'nvidia-smi failed: {exc!r}')

_print_gpu_info_via_nvidia_smi()

# Probe FlashInfer importability (doesn\'t init CUDA — just a Python import).
try:
    import flashinfer  # noqa: F401
    HAS_FLASHINFER = True
    print(f'FlashInfer is importable: version {getattr(flashinfer, "__version__", "?")}.')
except Exception as exc:
    HAS_FLASHINFER = False
    print(f'FlashInfer NOT importable: {exc!r}')
    print('  vLLM will auto-select FLASH_ATTN or TRITON_ATTN. The other speedups still apply.')

# ----- KEY DECISION: is FlashInfer actually usable on THIS GPU? -----
# FlashInfer 0.6.x ships kernels for SM 100 (datacenter Blackwell B100/B200)
# but NOT for SM 12.x (consumer/workstation Blackwell — RTX 5090, RTX PRO 6000).
# The "import" succeeds but the kernels don\'t load and the vLLM worker dies
# silently with "Failed core proc(s): {}" — exactly what your previous run hit.
# Auto-detect this case and suppress FlashInfer; vLLM will then pick FA4 which
# IS supported on SM 120 and is also extremely fast.
FLASHINFER_USABLE = HAS_FLASHINFER
if HAS_FLASHINFER and GPU_COMPUTE_CAPABILITY:
    cc_major = GPU_COMPUTE_CAPABILITY.split('.')[0]
    if cc_major == '12':
        FLASHINFER_USABLE = False
        print(f'\n⚠ FlashInfer is installed but SM {GPU_COMPUTE_CAPABILITY} ({GPU_NAME})')
        print('  is consumer/workstation Blackwell. FlashInfer 0.6.x does NOT have')
        print('  pre-built kernels for SM 12.x — only for SM 100 (datacenter B100/B200).')
        print('  Forcing FLASHINFER would crash the vLLM worker silently.')
        print('  vLLM will auto-select FlashAttention 4 instead, which is just as fast')
        print('  on Blackwell. All other speedups still apply.')
        print('  (Override with `FLASHINFER_USABLE = True` in Cell 6 if you have a')
        print('  custom-built FlashInfer with SM 12.x cubins.)')
print(f'\nFlashInfer effective: {FLASHINFER_USABLE}')


Imports OK
Running in Colab: True
BASE_DIR: /content/drive/MyDrive/swiss_law
CHECKPOINT_DIR: /content/drive/MyDrive/swiss_law/data/checkpoints
GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (compute capability 12.0); free=94.97 GiB / total=95.59 GiB; driver 580.82.07


FlashInfer is importable: version 0.6.9.

⚠ FlashInfer is installed but SM 12.0 (NVIDIA RTX PRO 6000 Blackwell Server Edition)
  is consumer/workstation Blackwell. FlashInfer 0.6.x does NOT have
  pre-built kernels for SM 12.x — only for SM 100 (datacenter B100/B200).
  Forcing FLASHINFER would crash the vLLM worker silently.
  vLLM will auto-select FlashAttention 4 instead, which is just as fast
  on Blackwell. All other speedups still apply.
  (Override with `FLASHINFER_USABLE = True` in Cell 6 if you have a
  custom-built FlashInfer with SM 12.x cubins.)

FlashInfer effective: False


In [2]:
# Cell 2 — Config (Blackwell-optimized; quality preserved)

@dataclass
class Config:
    # ---- Paths ----
    base_dir: str = str(BASE_DIR)
    data_dir: str = str(DATA_DIR)
    output_dir: str = str(OUTPUT_DIR)
    model_download_dir: str = str(MODEL_DIR / 'huggingface')
    local_scratch_dir: str = str(LOCAL_SCRATCH)

    # ---- Input ----
    # Targets-only input: 363,258 cards selected by identify_court_enrichment_targets.py.
    # This is ~15% of the original 2.4M-row corpus — the subset that actually needs LLM
    # semantic enrichment (the rest already have adequate static enrichment).
    input_jsonl: str = str(DATA_DIR / 'court_authority_cards_v4_target_cards.jsonl')
    fallback_input_jsonl: str = 'court_authority_cards_v4_target_cards.jsonl'

    # ---- Model ----
    model_name: str = 'Qwen/Qwen3-8B-AWQ'

    # ---- Slice ----
    start: int = 0
    limit: int = 0
    sample_random: bool = False
    random_seed: Optional[int] = 42
    min_text_chars: int = 80
    # 2400 chars matches the previous run; do NOT lower this without re-validating quality on long paragraphs.
    max_text_chars: int = 2400

    # ---- GPU / vLLM ----
    gpu_mode: str = 'single'           # 'single' or 'tp2'
    tensor_parallel_size: int = 1
    # 0.92 is safe on a dedicated 95 GB Blackwell. Drop to 0.88 if you see OOM during graph capture.
    gpu_memory_utilization: float = 0.92

    # 2048 covers system+schema (~270 tok) + truncated text up to ~1450 tok + output 320 tok with margin.
    # Lower than the previous 3072 → ~33% more KV-cache capacity for the same VRAM budget.
    max_model_len: int = 2048

    # 4× the previous concurrency. With FP8 KV + lower max_model_len, this fits comfortably.
    max_num_seqs: int = 384

    # Submit this many prompts per llm.generate() call. Must be >> max_num_seqs so the scheduler
    # always has work to swap in when a sequence finishes (kills the long-tail GPU-idle problem).
    submit_chunk: int = 2048

    enforce_eager: bool = False

    quantization: str = 'awq_marlin'
    disable_custom_all_reduce: bool = True

    # ---- KV cache ----
    # FP8 KV halves memory per token of context. Quality impact on this descriptor task is negligible.
    # Set to None to disable if you ever want a strict apples-to-apples baseline, or if you hit
    # an Engine-core-init error and want to isolate FP8 KV as the cause.
    kv_cache_dtype: Optional[str] = None

    # ---- Attention backend ----
    # In Cell 6, FLASHINFER is requested via the LLM(attention_backend=...) kwarg if FlashInfer
    # was installed successfully. Otherwise vLLM auto-selects (FA4 on Blackwell SM100+, FA3 on
    # Hopper SM90, FA2 elsewhere). The loader does NOT loop over backends — that pattern was
    # what produced the "LayerName already registered" crash on the previous run.

    # ---- Speculative decoding (n-gram, no extra weights) ----
    # JSON output reuses field names from the prompt (the schema in the system message), so prompt-lookup
    # n-gram speculation hits very often when it works.
    #
    # DEFAULT OFF on this GPU/stack. On SM 12.x (Blackwell) + AWQ Marlin + FP8 KV cache, vLLM's
    # n-gram path forces synchronous scheduling (you'll see "Async scheduling not supported with
    # ngram-based speculative decoding and will be disabled" in the log), and that synchronous
    # worker init path crashes silently with "Engine core initialization failed ... Failed core
    # proc(s): {}". Every other speedup in this config (FP8 KV, prefix caching, larger
    # max_num_seqs, lower max_model_len, FA4 backend) is unaffected and stays ON.
    #
    # If you want to try the +~30% spec-decoding bonus: flip this to True and rerun Cell 2 -> Cell 6.
    # If it loads, great. If it crashes the same way, leave it False — that's the known incompatibility.
    enable_ngram_speculation: bool = False
    speculative_num_tokens: int = 5
    speculative_ngram_min: int = 2
    speculative_ngram_max: int = 4

    # ---- Sampling / generation ----
    use_structured_outputs: bool = False  # leave OFF for strict apples-to-apples with previous run
    max_new_tokens: int = 320
    retry_max_new_tokens: int = 448
    max_retries: int = 1
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.0       # was 1.02; at T=0 the penalty has no quality benefit and costs decode time
    enable_thinking: bool = False

    # ---- Output / debug ----
    include_raw_output_on_success: bool = False

cfg = Config()

# Apply GPU mode before any vLLM import that would init CUDA.
if cfg.gpu_mode == 'single':
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
    cfg.tensor_parallel_size = 1
elif cfg.gpu_mode == 'tp2':
    os.environ.pop('CUDA_VISIBLE_DEVICES', None)
    cfg.tensor_parallel_size = 2
    cfg.disable_custom_all_reduce = True
else:
    raise ValueError("cfg.gpu_mode must be 'single' or 'tp2'")

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

base_dir = Path(cfg.base_dir)
data_dir = Path(cfg.data_dir)
out_dir = Path(cfg.output_dir)
local_scratch = Path(cfg.local_scratch_dir)
model_download_dir = Path(cfg.model_download_dir)
for d in [base_dir, data_dir, out_dir, model_download_dir, local_scratch]:
    d.mkdir(parents=True, exist_ok=True)

end_idx = cfg.start + cfg.limit - 1 if cfg.limit else -1
suffix = f'{cfg.start:07d}_{end_idx:07d}' if cfg.limit else f'{cfg.start:07d}_all'

# Drive paths (final destination)
output_jsonl = out_dir / f'court_llm_descriptors_{suffix}.jsonl'
output_preview_csv = out_dir / f'court_llm_descriptors_{suffix}_preview.csv'
output_failures_jsonl = out_dir / f'court_llm_descriptors_{suffix}_failures.jsonl'
output_metrics_json = out_dir / f'court_llm_descriptors_{suffix}_metrics.json'

# Local scratch paths (fast writes during the run)
local_jsonl = local_scratch / f'court_llm_descriptors_{suffix}.jsonl'
local_failures_jsonl = local_scratch / f'court_llm_descriptors_{suffix}_failures.jsonl'

print(json.dumps(asdict(cfg), indent=2, default=str))
print('Output JSONL (final on Drive):', output_jsonl)
print('Output JSONL (local hot-path):', local_jsonl)


{
  "base_dir": "/content/drive/MyDrive/swiss_law",
  "data_dir": "/content/drive/MyDrive/swiss_law/data",
  "output_dir": "/content/drive/MyDrive/swiss_law/outputs",
  "model_download_dir": "/content/drive/MyDrive/swiss_law/models/huggingface",
  "local_scratch_dir": "/content/drive/MyDrive/swiss_law/data/checkpoints",
  "input_jsonl": "/content/drive/MyDrive/swiss_law/data/court_authority_cards_v4_target_cards.jsonl",
  "fallback_input_jsonl": "court_authority_cards_v4_target_cards.jsonl",
  "model_name": "Qwen/Qwen3-8B-AWQ",
  "start": 0,
  "limit": 0,
  "sample_random": false,
  "random_seed": 42,
  "min_text_chars": 80,
  "max_text_chars": 2400,
  "gpu_mode": "single",
  "tensor_parallel_size": 1,
  "gpu_memory_utilization": 0.92,
  "max_model_len": 2048,
  "max_num_seqs": 384,
  "submit_chunk": 2048,
  "enforce_eager": false,
  "quantization": "awq_marlin",
  "disable_custom_all_reduce": true,
  "kv_cache_dtype": null,
  "enable_ngram_speculation": false,
  "speculative_num_token

In [7]:
# Cell 3 — Load target cards JSONL and select rows
#
# Input: court_authority_cards_v4_target_cards.jsonl (363,258 records).
# This file was produced by:
#   scripts/identify_court_enrichment_targets.py  (selects 363k from 2.4M cards)
#   scripts/extract_court_target_cards.py         (materializes full card payloads)
#
# Each card has:
#   citation                  — e.g. "BGE 130 I 26 E. 6.3.2"
#   text_excerpt_original     — full original-language paragraph
#   language                  — de | fr | it | unknown (deterministic)
#   legal_area                — pre-classified legal area (deterministic)
#   _enrichment_target.line_idx  — stable integer ID in the v4 cards corpus
#
# We use line_idx as the checkpoint key (stable across runs).

def resolve_input_path() -> Path:
    candidates = [
        Path(cfg.input_jsonl),
        DATA_DIR / cfg.fallback_input_jsonl,
        BASE_DIR / cfg.fallback_input_jsonl,
        Path('/content') / cfg.fallback_input_jsonl,
        Path('/kaggle/input') / cfg.fallback_input_jsonl,
        Path.cwd() / cfg.fallback_input_jsonl,
    ]
    for p in candidates:
        if p.exists():
            return p

    search_roots = [DATA_DIR, BASE_DIR, Path('/content'), Path('/kaggle/input')]
    for root in search_roots:
        if root.exists():
            for pat in ['**/court_authority_cards_v4_target_cards.jsonl']:
                found = sorted(root.glob(pat))
                if found:
                    return found[0]

    raise FileNotFoundError(
        'Could not find court_authority_cards_v4_target_cards.jsonl. Expected: '
        f'{DATA_DIR / cfg.fallback_input_jsonl}'
    )

input_path = resolve_input_path()
print('Using input:', input_path)
print(f'  size: {input_path.stat().st_size / 1024**2:.1f} MB')

# Stream the JSONL — extract only the fields we need (avoid loading 1 GB of unused metadata).
rows = []
skipped_no_text = 0
skipped_short = 0
with input_path.open(encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            card = json.loads(line)
        except Exception:
            continue
        text = (card.get('text_excerpt_original') or '').strip()
        citation = (card.get('citation') or '').strip()
        if not citation or not text:
            skipped_no_text += 1
            continue
        if len(text) < cfg.min_text_chars:
            skipped_short += 1
            continue
        # _source_row is the line_idx in the v4 cards corpus — stable, integer, unique.
        target_meta = card.get('_enrichment_target') or {}
        rows.append({
            '_source_row': int(target_meta.get('line_idx', line_num - 1)),
            'citation': citation,
            'text': text,
            'language': card.get('language', 'unknown'),
            'legal_area': card.get('legal_area', ''),
            '_text_len': len(text),
        })

print(f'Loaded: {len(rows):,} valid cards  '
      f'(skipped: {skipped_no_text:,} missing text/citation, {skipped_short:,} below min_text_chars)')

valid = pd.DataFrame(rows)
citation_col = 'citation'
text_col = 'text'

# Distribution sanity check
desc = valid['_text_len'].describe(percentiles=[0.5, 0.9, 0.95, 0.99])
print('Input text length distribution (chars):')
print(desc.to_string())
over = (valid['_text_len'] > cfg.max_text_chars).sum()
print(f"Rows exceeding max_text_chars={cfg.max_text_chars}: {over} / {len(valid)} "
      f"({100*over/max(len(valid),1):.2f}%)")
print('(Those rows get head+tail truncation; quality preserved.)')

# Language + legal_area mix (helpful sanity check)
print('Language mix:')
print(valid['language'].value_counts().to_string())

# Apply slice / sampling
if cfg.sample_random:
    pool = valid.iloc[cfg.start:] if cfg.start else valid
    work_df = pool.sample(n=min(cfg.limit, len(pool)) if cfg.limit else len(pool),
                          random_state=cfg.random_seed)
else:
    end = None if not cfg.limit else cfg.start + cfg.limit
    work_df = valid.iloc[cfg.start:end].copy()

# work_df already has _source_row column; reset_index is for stable iloc only.
work_df = work_df.reset_index(drop=True)
print(f'Selected rows: {len(work_df):,}')
display(work_df[['_source_row', citation_col, 'language', 'legal_area', '_text_len']].head(10))


Using input: /content/drive/MyDrive/swiss_law/data/court_authority_cards_v4_target_cards.jsonl
  size: 981.3 MB
Loaded: 363,258 valid cards  (skipped: 0 missing text/citation, 0 below min_text_chars)
Input text length distribution (chars):
count    363258.000000
mean        712.385742
std         403.284576
min         120.000000
50%         693.000000
90%        1203.000000
95%        1203.000000
99%        1203.000000
max        1203.000000
Rows exceeding max_text_chars=2400: 0 / 363258 (0.00%)
(Those rows get head+tail truncation; quality preserved.)
Language mix:
language
de         247255
fr          99097
it          14955
unknown      1951
Selected rows: 363,258


,_source_row,citation,language,legal_area,_text_len
0,1,BGE 139 I 2 E. 5.7,de,constitutional and public law,278
1,2,BGE 139 I 2 E. 6.3,de,constitutional and public law,314
2,3,BGE 145 I 1 E. 1.1.3,de,constitutional and public law,815
3,4,BGE 145 I 1 E. 6,de,constitutional and public law,327
4,5,BGE 145 I 1 E. 6.5.1,de,constitutional and public law,1203
5,6,BGE 145 I 1 E. 8.3,de,constitutional and public law,1093
6,7,BGE 142 I 1 E. 7.2.1,de,constitutional and public law,1203
7,8,BGE 136 I 1 E. 4.2,de,constitutional and public law,357
8,9,BGE 136 I 1 E. 4.4.4,de,constitutional and public law,1019
9,10,BGE 136 I 1 E. 5.4.3,de,constitutional and public law,1203


In [8]:
# Cell 4 — Schema and prompt
#
# Quality-preserving change vs. previous notebook:
#   The schema hint used to live at the END of the user message, AFTER citation+text.
#   Because citation+text vary per row, prefix caching could not include the schema.
#   We move the schema into the SYSTEM message (which is constant across all rows),
#   so the entire system+schema prefix is now cached and prefilled exactly once
#   per vLLM worker, not 2.4M times. This saves ~120 prompt tokens of compute per row.
#   The literal text the model sees is the same; only ordering changes.

DESCRIPTOR_KEYS = [
    'legal_area',
    'primary_domain',
    'secondary_domain',
    'legal_domain_path',
    'topic',
    'subtopic',
    'micro_topic',
    'concepts_en',
    'terms_original',
    'doctrinal_rule',
    'legal_test',
    'fact_pattern_tags',
    'procedural_context',
    'paragraph_role',
    'authority_role',
    'specificity_score',
]

ROLE_VALUES = {
    'holding', 'reasoning', 'facts', 'procedural_history', 'legal_standard',
    'application', 'citation', 'costs', 'notification', 'disposition', 'neutral'
}

LLM_SCHEMA_HINT = {
    'legal_area': 'broad area, <=5 words',
    'primary_domain': '<=5 words',
    'secondary_domain': '<=7 words',
    'legal_domain_path': ['2-4 labels'],
    'topic': '<=7 words',
    'subtopic': '<=9 words',
    'micro_topic': '<=12 words; most specific issue',
    'concepts_en': ['3-5 English legal concepts'],
    'terms_original': ['3-6 exact source-language legal terms'],
    'doctrinal_rule': 'empty unless paragraph states rule; <=18 words',
    'legal_test': 'empty unless test/standard; <=16 words',
    'fact_pattern_tags': ['0-4 concrete tags'],
    'procedural_context': '<=8 words',
    'paragraph_role': 'holding|reasoning|facts|procedural_history|legal_standard|application|citation|costs|notification|disposition|neutral',
    'authority_role': ['0-2 labels'],
    'specificity_score': '0..1',
}

_SCHEMA_JSON = json.dumps(LLM_SCHEMA_HINT, ensure_ascii=False, separators=(',', ':'))

# System message now carries instructions AND the schema. Constant across all 2.4M rows
# → fully prefix-cacheable.
SYSTEM_PROMPT = f'''You are a Swiss legal descriptor extractor.

Return exactly one compact JSON object with exactly this shape (all keys present):
{_SCHEMA_JSON}

Rules:
- Use English for classification fields.
- Use exact German/French/Italian terms for terms_original.
- If the paragraph is factual/procedural/boilerplate, keep doctrinal_rule and legal_test empty.
- Prefer precise legal descriptors over generic words.
- Keep all strings and arrays short.
- Do not generate user questions or summaries.
- Do not extract statute anchors, case anchors, outcomes, retrieval views, or quality flags.
- JSON only.'''

# User message now contains ONLY the variable per-row content.
USER_TEMPLATE = '''Citation: {citation}

Text:
{text}

Return JSON only.'''

def trim_text(text: str, max_chars: int) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head
    return text[:head].rstrip() + ' ... [TRUNCATED] ... ' + text[-tail:].lstrip()

def build_user_prompt(citation: str, text: str) -> str:
    return USER_TEMPLATE.format(
        citation=str(citation),
        text=trim_text(text, cfg.max_text_chars),
    )


In [9]:
# Cell 5 — JSON parsing and descriptor normalization
#
# UNCHANGED from previous notebook. This is the quality contract: the same
# input JSON produces the same normalized output as before. Do not edit unless
# you intentionally want to change validation behavior.

def extract_json_object(raw: str) -> dict[str, Any]:
    if raw is None:
        raise ValueError('empty model output')
    s = str(raw).strip()
    s = re.sub(r'^\s*```(?:json)?\s*', '', s, flags=re.I)
    s = re.sub(r'\s*```\s*$', '', s)
    s = re.sub(r'<think>.*?</think>', '', s, flags=re.I | re.S).strip()

    start = s.find('{')
    if start < 0:
        raise ValueError(f'no JSON object start found: {s[:300]}')

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    candidate = s[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        candidate = re.sub(r',\s*([}\]])', r'\1', candidate)
                        try:
                            return json.loads(candidate)
                        except Exception:
                            return ast.literal_eval(candidate)
    raise ValueError(f'no balanced JSON object found: {s[:700]}')


def clean_str(x: Any, max_chars: int = 240) -> str:
    s = re.sub(r'\s+', ' ', str(x or '')).strip()
    return s[:max_chars].rstrip()


def clean_list(x: Any, max_items: int, max_chars: int = 80) -> list[str]:
    if x is None:
        return []
    if isinstance(x, str):
        x = [x]
    if not isinstance(x, (list, tuple, set)):
        return []
    out, seen = [], set()
    for item in x:
        s = clean_str(item, max_chars=max_chars)
        if not s:
            continue
        key = s.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_items:
            break
    return out


def normalize_descriptor(obj: dict[str, Any]) -> dict[str, Any]:
    d = {}
    d['legal_area'] = clean_str(obj.get('legal_area'), 80)
    d['primary_domain'] = clean_str(obj.get('primary_domain'), 80)
    d['secondary_domain'] = clean_str(obj.get('secondary_domain'), 100)
    d['legal_domain_path'] = clean_list(obj.get('legal_domain_path'), 4, 60)
    d['topic'] = clean_str(obj.get('topic'), 100)
    d['subtopic'] = clean_str(obj.get('subtopic'), 120)
    d['micro_topic'] = clean_str(obj.get('micro_topic'), 160)
    d['concepts_en'] = clean_list(obj.get('concepts_en'), 5, 70)
    d['terms_original'] = clean_list(obj.get('terms_original'), 6, 100)
    d['doctrinal_rule'] = clean_str(obj.get('doctrinal_rule'), 260)
    d['legal_test'] = clean_str(obj.get('legal_test'), 220)
    d['fact_pattern_tags'] = clean_list(obj.get('fact_pattern_tags'), 4, 70)
    d['procedural_context'] = clean_str(obj.get('procedural_context'), 120)

    role = clean_str(obj.get('paragraph_role'), 60).lower().replace(' ', '_').replace('-', '_')
    d['paragraph_role'] = role if role in ROLE_VALUES else 'neutral'
    d['authority_role'] = clean_list(obj.get('authority_role'), 2, 60)
    try:
        d['specificity_score'] = max(0.0, min(1.0, float(obj.get('specificity_score', 0))))
    except Exception:
        d['specificity_score'] = 0.0

    forbidden = {
        'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
        'outcome_signal', 'query_phrases_en', 'natural_language_queries', 'legal_question',
        'summary_en', 'english_summary', 'enrichment_quality', 'anchor_quality_flags',
    }
    for k in forbidden:
        d.pop(k, None)
    return d


def empty_descriptor(error: str = '') -> dict[str, Any]:
    return {
        'legal_area': '',
        'primary_domain': '',
        'secondary_domain': '',
        'legal_domain_path': [],
        'topic': '',
        'subtopic': '',
        'micro_topic': '',
        'concepts_en': [],
        'terms_original': [],
        'doctrinal_rule': '',
        'legal_test': '',
        'fact_pattern_tags': [],
        'procedural_context': '',
        'paragraph_role': 'neutral',
        'authority_role': [],
        'specificity_score': 0.0,
        '_descriptor_error': error[:500],
    }


In [10]:
# Cell 6 — Load vLLM (single attempt, no fallback re-imports)
#
# Why single-attempt with no re-import loop:
#   The previous loop did `del sys.modules['vllm.*']` and re-imported vLLM under
#   different env vars. That triggered a torch op double-registration crash:
#     RuntimeError: Type 'vllm.utils.torch_utils.LayerName' is already registered
#                   as an opaque type
#   Once vLLM\'s C++ extensions register their ops in a process, you cannot cleanly
#   re-init them. So we pick the right backend ONCE, build LLM ONCE, and surface
#   any failure clearly so it can be diagnosed.
#
# Backend selection:
#   vLLM 0.11+ deprecated VLLM_ATTENTION_BACKEND in favor of the LLM constructor
#   kwarg `attention_backend` (or `attention_config=AttentionConfig(backend=...)`).
#   The Unknown-environment-variable warning you saw confirms your build is on
#   the new path. We probe LLM.__init__ for `attention_backend`; if present we
#   pass FLASHINFER there. If not, we fall back to setting the env var (older builds).
#   If FlashInfer is unavailable, we let vLLM auto-select (Blackwell defaults to
#   FlashAttention 4 / TRITON_ATTN — both still benefit from FP8 KV + larger
#   max_num_seqs + continuous batching).

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
import inspect

print('Loading tokenizer:', cfg.model_name)

tokenizer = AutoTokenizer.from_pretrained(
    cfg.model_name,
    trust_remote_code=True,
    cache_dir=cfg.model_download_dir,
)

# Sanity-check the chat template + measure the cached prefix.
_probe_messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'probe'},
]
try:
    _probe_text = tokenizer.apply_chat_template(
        _probe_messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=cfg.enable_thinking,
    )
except TypeError:
    _probe_text = tokenizer.apply_chat_template(
        _probe_messages, tokenize=False, add_generation_prompt=True,
    )
_probe_tokens = tokenizer(_probe_text, return_tensors=None, add_special_tokens=False)['input_ids']
print(f'System+schema prefix size: ~{len(_probe_tokens)} tokens (this is the prefix-cached portion).')

# ----- Build kwargs -----
# vLLM\'s LLM constructor accepts many fields via **kwargs that don\'t appear in
# inspect.signature() — they\'re forwarded to EngineArgs. Our previous probe
# was too conservative and silently dropped FP8 KV / prefix caching / spec
# decoding. We now pass them DIRECTLY; if a given vLLM build truly doesn\'t
# accept one of them, the LLM constructor will raise a clear TypeError, which
# we surface in the diagnostic block below.
sig_params = set(inspect.signature(LLM).parameters.keys())
print(f'vLLM LLM signature ({len(sig_params)} explicit params); attention_config={"attention_config" in sig_params}')

# Decide whether to actually use FlashInfer (set by Cell 1\'s SM-based check).
# `FLASHINFER_USABLE` is False for SM 12.x even when import succeeded.
desired_backend = 'FLASHINFER' if FLASHINFER_USABLE else None

llm_kwargs = dict(
    model=cfg.model_name,
    trust_remote_code=True,
    tensor_parallel_size=cfg.tensor_parallel_size,
    gpu_memory_utilization=cfg.gpu_memory_utilization,
    max_model_len=cfg.max_model_len,
    max_num_seqs=cfg.max_num_seqs,
    enforce_eager=cfg.enforce_eager,
    disable_custom_all_reduce=cfg.disable_custom_all_reduce,
    disable_log_stats=True,
    download_dir=cfg.model_download_dir,
    quantization=cfg.quantization,
)

# Pass these directly — they go through **kwargs to EngineArgs in vLLM 0.11+.
if cfg.kv_cache_dtype is not None:
    llm_kwargs['kv_cache_dtype'] = cfg.kv_cache_dtype

llm_kwargs['enable_prefix_caching'] = True

# Attention backend: only set it if FlashInfer is actually usable on this GPU.
backend_method = 'auto-select (vLLM picks FA4 on Blackwell SM 12.x)'
if desired_backend is not None:
    if 'attention_backend' in sig_params:
        llm_kwargs['attention_backend'] = desired_backend
        backend_method = f'attention_backend kwarg = {desired_backend}'
    elif 'attention_config' in sig_params:
        try:
            from vllm.config import AttentionConfig
            llm_kwargs['attention_config'] = AttentionConfig(backend=desired_backend)
            backend_method = f'attention_config = AttentionConfig(backend={desired_backend})'
        except Exception as exc:
            print(f'  AttentionConfig import failed ({exc!r}); using env var fallback')
            os.environ['VLLM_ATTENTION_BACKEND'] = desired_backend
            backend_method = f'env VLLM_ATTENTION_BACKEND = {desired_backend} (deprecated path)'
    else:
        os.environ['VLLM_ATTENTION_BACKEND'] = desired_backend
        backend_method = f'env VLLM_ATTENTION_BACKEND = {desired_backend} (deprecated path)'
    os.environ.setdefault('VLLM_USE_FLASHINFER_SAMPLER', '1')

# Speculative decoding — pass directly; if not supported, vLLM ignores or errors.
spec_active = False
if cfg.enable_ngram_speculation:
    llm_kwargs['speculative_config'] = {
        'method': 'ngram',
        'num_speculative_tokens': cfg.speculative_num_tokens,
        'prompt_lookup_max': cfg.speculative_ngram_max,
        'prompt_lookup_min': cfg.speculative_ngram_min,
    }
    spec_active = True

print('\nvLLM init plan:')
print(f'  attention backend: {backend_method}')
print(f'  KV cache dtype:    {llm_kwargs.get("kv_cache_dtype", "default fp16")}')
print(f'  prefix caching:    {llm_kwargs.get("enable_prefix_caching", False)}')
print(f'  speculative dec.:  {"n-gram" if spec_active else "off"}')
print(f'  max_model_len:     {cfg.max_model_len}')
print(f'  max_num_seqs:      {cfg.max_num_seqs}')
print(f'  gpu_mem_util:      {cfg.gpu_memory_utilization}')

# ----- Single load attempt -----
print('\nInitializing vLLM (this can take a minute)...')
try:
    llm = LLM(**llm_kwargs)
except Exception as exc:
    print(f'\n✗ vLLM load FAILED: {type(exc).__name__}: {exc}')
    print('\nDiagnostics:')
    print('  1. RESTART THE COLAB RUNTIME and re-run from Cell 0 → Cell 1 → ... → Cell 6.')
    print('     "Engine core initialization failed ... Failed core proc(s): {}" usually means')
    print('     the vLLM worker subprocess crashed silently. Most common causes on this GPU:')
    print(f'       (a) attention backend incompatibility — your GPU is SM {GPU_COMPUTE_CAPABILITY}.')
    print('           If FLASHINFER_USABLE was True, set it False above and rerun this cell only.')
    print('       (b) FP8 KV cache rejected by the driver/runtime combo.')
    print('           Set cfg.kv_cache_dtype = None in Cell 2 and rerun Cell 2 → Cell 6.')
    print('       (c) speculative_config not accepted in this vLLM build.')
    print('           Set cfg.enable_ngram_speculation = False in Cell 2.')
    print('  2. Try a slimmer kwargs dict (uncomment in this cell): drop kv_cache_dtype,')
    print('     enable_prefix_caching, speculative_config — load with bare config first to')
    print('     confirm the model+backend works at all, then re-add features one at a time.')
    print('  3. As a last resort, set in Cell 2: cfg.gpu_memory_utilization = 0.85 and')
    print('     cfg.max_num_seqs = 256 to reduce memory pressure during graph capture.')
    raise

# Identify the actual backend vLLM picked (post-load).
chosen_backend = desired_backend or 'auto'
chosen_quantization = cfg.quantization
chosen_speculation = spec_active

print('\n' + '=' * 60)
print('vLLM loaded successfully.')
print(f'  attention backend (requested): {chosen_backend}')
print(f'  quantization:                  {chosen_quantization}')
print(f'  KV cache dtype:                {cfg.kv_cache_dtype}')
print(f'  speculative decoding:          {chosen_speculation}')
print('=' * 60)

# Now it's safe to import torch for post-load metrics (vLLM has already initialized CUDA).
try:
    import torch as _t
    globals()['torch'] = _t
    if _t.cuda.is_available():
        for i in range(_t.cuda.device_count()):
            free, total = _t.cuda.mem_get_info(i)
            print(f'After vLLM load GPU {i}: free={free/1024**3:.2f} GiB total={total/1024**3:.2f} GiB')
except Exception:
    pass


Loading tokenizer: Qwen/Qwen3-8B-AWQ
System+schema prefix size: ~315 tokens (this is the prefix-cached portion).
vLLM LLM signature (37 explicit params); attention_config=True

vLLM init plan:
  attention backend: auto-select (vLLM picks FA4 on Blackwell SM 12.x)
  KV cache dtype:    default fp16
  prefix caching:    True
  speculative dec.:  off
  max_model_len:     2048
  max_num_seqs:      384
  gpu_mem_util:      0.92

Initializing vLLM (this can take a minute)...
INFO 05-04 15:07:58 [utils.py:233] non-default args: {'trust_remote_code': True, 'download_dir': '/content/drive/MyDrive/swiss_law/models/huggingface', 'max_model_len': 2048, 'enable_prefix_caching': True, 'max_num_seqs': 384, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'disable_custom_all_reduce': True, 'model': 'Qwen/Qwen3-8B-AWQ'}


config.json: 0.00B [00:00, ?B/s]

INFO 05-04 15:08:07 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-04 15:08:07 [nixl_utils.py:34] NIXL is not available
WARNING 05-04 15:08:07 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-04 15:08:07 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-04 15:08:07 [model.py:1680] Using max model len 2048
INFO 05-04 15:08:07 [awq_marlin.py:252] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 05-04 15:08:07 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=16384.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


vLLM loaded successfully.
  attention backend (requested): auto
  quantization:                  awq_marlin
  KV cache dtype:                None
  speculative decoding:          False
After vLLM load GPU 0: free=6.98 GiB total=94.97 GiB


In [11]:
# Cell 7 — Generation helpers
#
# Two helpers:
#   render_prompt(citation, text, ...) — builds the chat-templated prompt string.
#   generate_raw_safe(prompts, max_tokens) — submits the WHOLE list to vLLM in
#       one llm.generate() call. vLLM continuous batching handles scheduling
#       across max_num_seqs slots. On OOM, recursively splits the list.
#
# `generate_descriptor()` handles the per-row retry path for parse failures only;
# the main loop never calls it for the happy path (it pre-batches everything).

def render_prompt(citation: str, text: str, repair: bool = False, bad_output: str = '', error: str = '') -> str:
    user_prompt = build_user_prompt(citation, text)
    if repair:
        user_prompt = f"""The previous output was invalid JSON.

Parser error:
{error}

Previous output:
{bad_output[:1400]}

Repair by returning exactly one complete compact JSON object using the same schema.
Do not add questions, summaries, anchors, outcomes, or retrieval views.

{user_prompt}"""

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=cfg.enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_raw(prompts: list[str], max_tokens: int) -> list[str]:
    params = SamplingParams(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=max_tokens,
        repetition_penalty=cfg.repetition_penalty,
    )
    # use_tqdm=True only for the big main-loop calls; the parent loop drives its own bar
    # via len(prompts), so we pass use_tqdm=False here and emit one outer bar.
    outputs = llm.generate(prompts, sampling_params=params, use_tqdm=False)
    return [out.outputs[0].text if out.outputs else '' for out in outputs]


def generate_raw_safe(prompts: list[str], max_tokens: int, min_split: int = 1) -> list[str]:
    """Generate a batch; on OOM/runtime failure, recursively split."""
    try:
        return generate_raw(prompts, max_tokens)
    except Exception as exc:
        if len(prompts) <= min_split:
            raise
        mid = len(prompts) // 2
        print(f'  Batch of {len(prompts)} failed ({type(exc).__name__}); splitting into {mid}+{len(prompts)-mid}')
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
        left = generate_raw_safe(prompts[:mid], max_tokens, min_split=min_split)
        right = generate_raw_safe(prompts[mid:], max_tokens, min_split=min_split)
        return left + right


def parse_or_retry(citation: str, text: str, raw: str | None) -> tuple[dict[str, Any], dict[str, Any]]:
    """Parse a raw output. If it fails, run the repair retry path (per-row)."""
    attempts = []
    for attempt in range(cfg.max_retries + 1):
        try:
            if raw is None:
                raw = generate_raw_safe([render_prompt(citation, text)], cfg.max_new_tokens)[0]
            obj = extract_json_object(raw)
            desc = normalize_descriptor(obj)
            return desc, {
                'status': 'ok' if attempt == 0 else 'ok_after_retry',
                'attempt_count': attempt + 1,
                'error': None,
                'raw_output': raw if cfg.include_raw_output_on_success else None,
            }
        except Exception as exc:
            err = repr(exc)
            attempts.append({'attempt': attempt + 1, 'error': err, 'raw_output': (raw or '')[:1400]})
            if attempt >= cfg.max_retries:
                return empty_descriptor(err), {
                    'status': 'failed_descriptor_parse',
                    'attempt_count': attempt + 1,
                    'error': err,
                    'attempts': attempts,
                    'raw_output': raw,
                }
            raw = generate_raw_safe(
                [render_prompt(citation, text, repair=True, bad_output=raw or '', error=err)],
                cfg.retry_max_new_tokens,
            )[0]


# Tiny warm-up so the first real chunk doesn't pay CUDA-graph capture latency.
print('Warm-up generation (16 short prompts)...')
_warm_prompts = [render_prompt('warmup', 'Warm-up paragraph for CUDA graph capture and FlashInfer kernel selection.')] * 16
_t = time.time()
_ = generate_raw_safe(_warm_prompts, max_tokens=64)
print(f'  done in {time.time()-_t:.2f}s')


Warm-up generation (16 short prompts)...
  done in 23.59s


In [12]:
# Cell 8 — Run descriptor extraction (mega-batched, local-disk hot path)
#
# Submission strategy:
#   For each chunk of cfg.submit_chunk prompts (default 2048), we call llm.generate()
#   ONCE with the whole list. vLLM internally schedules up to cfg.max_num_seqs (384)
#   sequences concurrently and immediately swaps in new ones from the queue as
#   sequences finish. This eliminates the long-tail GPU-idle that kills the previous
#   batch_size=64 setup.
#
# Output:
#   Hot-path JSONL is written to local SSD (/content/scratch_outputs/...). At the end
#   of the run we copy to Drive. This avoids paying network round-trip latency on
#   every flush.

records_for_preview = []
failures = []
status_counter = Counter()
t0 = time.time()
processed = 0
prompt_token_total = 0
output_token_total = 0

# ── Checkpoint resume ────────────────────────────────────────────────────────
# The local JSONL is the checkpoint. On a fresh run it does not exist → we
# write it from scratch. On a resume, we scan it for _source_row values that
# are already done and skip them. A partial last line (crash mid-write) is
# silently dropped by the try/except.
done_rows: set[int] = set()
if local_jsonl.exists():
    print(f'Checkpoint found: {local_jsonl}')
    with local_jsonl.open(encoding='utf-8') as _ckpt:
        for _line in _ckpt:
            _line = _line.strip()
            if not _line:
                continue
            try:
                done_rows.add(int(json.loads(_line)['_source_row']))
            except Exception:
                pass
    print(f'  → {len(done_rows):,} rows already done; will skip.')
else:
    print('No checkpoint found — starting fresh.')

remaining_df = work_df[~work_df['_source_row'].isin(done_rows)].reset_index(drop=True)
print(f'Rows remaining: {len(remaining_df):,} / {len(work_df):,} total ({len(done_rows):,} skipped).')
# ─────────────────────────────────────────────────────────────────────────────

# We need access to vLLM's RequestOutput tokens for accurate metrics.
# vLLM 0.10+ exposes prompt_token_ids and outputs[0].token_ids on each RequestOutput.
def _generate_with_metrics(prompts: list[str], max_tokens: int):
    params = SamplingParams(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=max_tokens,
        repetition_penalty=cfg.repetition_penalty,
    )
    return llm.generate(prompts, sampling_params=params, use_tqdm=False)


def _generate_with_metrics_safe(prompts: list[str], max_tokens: int, min_split: int = 1):
    try:
        return _generate_with_metrics(prompts, max_tokens)
    except Exception as exc:
        if len(prompts) <= min_split:
            raise
        mid = len(prompts) // 2
        print(f'  Batch of {len(prompts)} failed ({type(exc).__name__}); splitting into {mid}+{len(prompts)-mid}')
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
        left = _generate_with_metrics_safe(prompts[:mid], max_tokens, min_split=min_split)
        right = _generate_with_metrics_safe(prompts[mid:], max_tokens, min_split=min_split)
        return left + right


N = len(remaining_df)
SUBMIT = cfg.submit_chunk
print(f'Processing {N:,} remaining rows in chunks of {SUBMIT} (max_num_seqs={cfg.max_num_seqs}; {len(done_rows):,} already done).')

with local_jsonl.open('a', encoding='utf-8') as out_f:
    pbar = tqdm(total=N, desc='LLM descriptor extraction', smoothing=0.05)

    for start in range(0, N, SUBMIT):
        end = min(start + SUBMIT, N)
        chunk = remaining_df.iloc[start:end]

        # Build all prompts for this chunk.
        row_objs = []
        prompts = []
        for _, row in chunk.iterrows():
            citation = str(row[citation_col])
            text = str(row[text_col])
            row_objs.append({
                '_source_row': int(row['_source_row']),
                'citation': citation,
                'text': text,
                'language': row.get('language', 'unknown') if hasattr(row, 'get') else (row['language'] if 'language' in row.index else 'unknown'),
                'legal_area_static': row.get('legal_area', '') if hasattr(row, 'get') else (row['legal_area'] if 'legal_area' in row.index else ''),
            })
            prompts.append(render_prompt(citation, text))

        chunk_t = time.time()
        try:
            req_outputs = _generate_with_metrics_safe(prompts, cfg.max_new_tokens)
        except Exception as exc:
            print(f'Chunk {start}-{end} failed completely; falling back to per-row processing: {exc!r}')
            req_outputs = [None] * len(row_objs)

        for row_obj, req_out in zip(row_objs, req_outputs):
            if req_out is None:
                # Will trigger the per-row retry path inside parse_or_retry(raw=None).
                raw = None
            else:
                raw = req_out.outputs[0].text if req_out.outputs else ''
                # Token accounting for metrics
                try:
                    prompt_token_total += len(req_out.prompt_token_ids or [])
                    if req_out.outputs:
                        output_token_total += len(req_out.outputs[0].token_ids or [])
                except Exception:
                    pass

            desc, gen = parse_or_retry(row_obj['citation'], row_obj['text'], raw)
            rec = {
                '_source_row': row_obj['_source_row'],
                'citation': row_obj['citation'],
                'text': row_obj['text'],
                'language': row_obj['language'],
                'legal_area_static': row_obj['legal_area_static'],
                'llm_enrichment': desc,
                'llm_generation': {
                    'model': cfg.model_name,
                    'method': 'minimal_descriptor_only',
                    'attention_backend': chosen_backend,
                    'kv_cache_dtype': cfg.kv_cache_dtype,
                    'speculative': chosen_speculation,
                    **gen,
                },
            }
            out_f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            processed += 1
            status_counter[gen['status']] += 1
            if len(records_for_preview) < 1000:
                records_for_preview.append(rec)
            if gen['status'].startswith('failed'):
                failures.append(rec)

        out_f.flush()  # one flush per chunk (every ~2048 rows), not per batch of 64
        pbar.update(end - start)

        rate = (end) / max(time.time() - t0, 1e-9)
        chunk_rate = (end - start) / max(time.time() - chunk_t, 1e-9)
        pbar.set_postfix(
            chunk_rps=f'{chunk_rate:.1f}',
            avg_rps=f'{rate:.1f}',
            ok=status_counter['ok'],
            failed=status_counter['failed_descriptor_parse'],
        )

    pbar.close()

elapsed = time.time() - t0
print(f'\nFinished {processed} rows in {elapsed:.1f}s = {processed/elapsed:.2f} rows/sec')
print(f'Prompt tokens total:  {prompt_token_total:,}  (avg {prompt_token_total/max(processed,1):.0f}/row)')
print(f'Output tokens total:  {output_token_total:,}  (avg {output_token_total/max(processed,1):.0f}/row)')

# ----- Copy local hot-path JSONL to Drive -----
print(f'\nCopying {local_jsonl} → {output_jsonl} ...')
shutil.copy2(local_jsonl, output_jsonl)
print('  done.')

# ----- Failures, preview, metrics -----
if failures:
    with local_failures_jsonl.open('w', encoding='utf-8') as f:
        for rec in failures:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    shutil.copy2(local_failures_jsonl, output_failures_jsonl)
else:
    if output_failures_jsonl.exists():
        output_failures_jsonl.unlink()

preview_rows = []
for rec in records_for_preview:
    e = rec['llm_enrichment']
    g = rec['llm_generation']
    preview_rows.append({
        '_source_row': rec['_source_row'],
        'citation': rec['citation'],
        'status': g['status'],
        'legal_area': e.get('legal_area'),
        'primary_domain': e.get('primary_domain'),
        'secondary_domain': e.get('secondary_domain'),
        'topic': e.get('topic'),
        'subtopic': e.get('subtopic'),
        'micro_topic': e.get('micro_topic'),
        'concepts_en': ' | '.join(e.get('concepts_en', [])),
        'terms_original': ' | '.join(e.get('terms_original', [])),
        'doctrinal_rule': e.get('doctrinal_rule'),
        'legal_test': e.get('legal_test'),
        'fact_pattern_tags': ' | '.join(e.get('fact_pattern_tags', [])),
        'procedural_context': e.get('procedural_context'),
        'paragraph_role': e.get('paragraph_role'),
        'authority_role': ' | '.join(e.get('authority_role', [])),
        'specificity_score': e.get('specificity_score'),
    })
preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(output_preview_csv, index=False)

metrics = {
    'start': cfg.start,
    'limit': cfg.limit,
    'selected_rows': len(work_df),
    'written_rows_this_session': processed,
    'total_rows_in_file': len(done_rows) + processed,
    'failures': len(failures),
    'elapsed_seconds': elapsed,
    'rows_per_second': processed / max(elapsed, 1e-9),
    'prompt_tokens_total': prompt_token_total,
    'output_tokens_total': output_token_total,
    'tokens_per_second': (prompt_token_total + output_token_total) / max(elapsed, 1e-9),
    'output_tokens_per_second': output_token_total / max(elapsed, 1e-9),
    'attention_backend': chosen_backend,
    'quantization': chosen_quantization,
    'kv_cache_dtype': cfg.kv_cache_dtype,
    'speculative': chosen_speculation,
    'status_counts': dict(status_counter),
    'output_jsonl': str(output_jsonl),
    'output_preview_csv': str(output_preview_csv),
    'output_failures_jsonl': str(output_failures_jsonl) if failures else None,
    'config': {k: (str(v) if not isinstance(v, (int, float, bool, str, list, dict, type(None))) else v)
               for k, v in asdict(cfg).items()},
}
output_metrics_json.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(metrics, indent=2, default=str))
display(preview_df.head(50))


No checkpoint found — starting fresh.
Rows remaining: 363,258 / 363,258 total (0 skipped).
Processing 363,258 remaining rows in chunks of 2048 (max_num_seqs=384; 0 already done).


LLM descriptor extraction:   0%|          | 0/363258 [00:00<?, ?it/s]


Finished 363258 rows in 15895.0s = 22.85 rows/sec
Prompt tokens total:  207,978,225  (avg 573/row)
Output tokens total:  80,692,838  (avg 222/row)

Copying /content/drive/MyDrive/swiss_law/data/checkpoints/court_llm_descriptors_0000000_all.jsonl → /content/drive/MyDrive/swiss_law/outputs/court_llm_descriptors_0000000_all.jsonl ...
  done.
{
  "start": 0,
  "limit": 0,
  "selected_rows": 363258,
  "written_rows_this_session": 363258,
  "total_rows_in_file": 363258,
  "failures": 1,
  "elapsed_seconds": 15895.011837005615,
  "rows_per_second": 22.853584742496952,
  "prompt_tokens_total": 207978225,
  "output_tokens_total": 80692838,
  "tokens_per_second": 18161.11028794184,
  "output_tokens_per_second": 5076.613897961169,
  "attention_backend": "auto",
  "quantization": "awq_marlin",
  "kv_cache_dtype": null,
  "speculative": false,
  "status_counts": {
    "ok": 363212,
    "ok_after_retry": 45,
    "failed_descriptor_parse": 1
  },
  "output_jsonl": "/content/drive/MyDrive/swiss_law/o

,_source_row,citation,status,legal_area,primary_domain,secondary_domain,topic,subtopic,micro_topic,concepts_en,terms_original,doctrinal_rule,legal_test,fact_pattern_tags,procedural_context,paragraph_role,authority_role,specificity_score
0,1,BGE 139 I 2 E. 5.7,ok,Constitutional law,Local planning,Land use regulation,Zoning changes,Initiative for new zoning,Initiative to rezone land for public parks,Zoning | Initiative | Land use | Public parks ...,Volksabstimmung | Umzonung | Parkanlagen | Haf...,,,Initiative | Zoning change | Public parks | La...,Initiative adoption,facts,Swiss Confederation | Local authority,0.2
1,2,BGE 139 I 2 E. 6.3,ok,Constitutional law,Voting rights,Public assembly procedures,Free expression,Sincerity of intent,Sincerity of intent in public assembly documen...,Free expression | Sincerity | Public assembly ...,freie Willensbildung | Sachlichkeit | Gemeinde...,,,public assembly | documentation | sincerity,Constitutional guarantee application,reasoning,Swiss Federal Court,0.8
2,3,BGE 145 I 1 E. 1.1.3,ok,Constitutional law,Federal governance,Federal legislative process,Judicial review limits,Unreviewability of federal acts,Unreviewability of federal legislative explana...,Judicial review | Federal legislation | Unrevi...,Bundesversammlung | Bundesrat | Abstimmungserl...,,,Federal legislation | Unreviewability | Legisl...,Constitutional interpretation,reasoning,Bundesgericht,0.8
3,4,BGE 145 I 1 E. 6,ok,Constitutional law,Freedom of speech,Political campaign regulations,Campaign interference,Unlawful political campaign involvement,Unlawful interference in political campaign by...,Freedom of speech | Political campaign | Publi...,Abstimmungskampf | Geldspielgesetz | KdK | Fac...,,,Political campaign | Public authority | Consti...,Constitutional complaint,facts,Public authority | Constitutional court,0.8
4,5,BGE 145 I 1 E. 6.5.1,ok,Constitutional law,Federalism,Interference with referendums,Referendum participation,Role of cantonal authorities,Cantonal influence on federal referendum outcomes,Federalism | Referendum | Cantonal authority |...,Stimmberechtigte | Kantonsregierung | Vorlage ...,,,Federal referendums | Cantonal intervention | ...,Referendum process,reasoning,Federal authorities | Cantonal authorities,0.8
5,6,BGE 145 I 1 E. 8.3,ok,Election law,Campaign finance,Public funding of elections,Campaign expenditure,Limits on campaign spending,Reasonableness of campaign finance expenditures,Campaign finance | Public funding | Reasonable...,Abstimmungskampf | finanzieller Betrag | Ausga...,,,Campaign finance | Election | Reasonableness |...,Election dispute,reasoning,Swiss Federal Court,0.8
6,7,BGE 142 I 1 E. 7.2.1,ok,Constitutional law,Fundamental rights,Right to assistance in need,Basic rights and assistance,Minimum assistance in need,Scope of constitutional right to assistance in...,Basic rights | Social assistance | Constitutio...,Art. 12 BV | Nothilfe | verfassungsrechtlich g...,,,Constitutional interpretation | Need-based aid...,Constitutional interpretation,reasoning,Swiss Federal Court | Constitutional law,0.8
7,8,BGE 136 I 1 E. 4.2,ok,Constitutional law,Equality rights,Discrimination based on race,Race-based regulations,Dog breed classification and equality,Race-based dog breed regulations and constitut...,Equality before the law | Discrimination | Con...,Rechtsgleichheitsgebot | Rassetypen | Gestaltu...,,,Constitutional review | Race discrimination | ...,Supreme court decision,reasoning,Swiss Federal Supreme Court,0.8
8,9,BGE 136 I 1 E. 4.4.4,ok,Federalism and equality,Federal constitutional law,Federal-state relations,Federalism and equality,Differentiation of cantonal regulations,Cantonal differences in dog risk assessment,Federalism | Equality | Cantonal autonomy | Co...,Rechtsgleichheitsprinzip | föderalistische Sta...,,,Cantonal differences | Federal structure | Equ...,Constitutional interpretation,reasoning,Federal Supreme Court | Constitutional law,0.8
9,10,BGE 136 I 1 E. 5.4.3,ok,Animal welfar

In [13]:
# Cell 9 — QC: verify this is raw descriptor output only

FORBIDDEN = {
    'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
    'outcome_signal', 'query_phrases_en', 'natural_language_queries', 'legal_question',
    'summary_en', 'english_summary', 'enrichment_quality', 'anchor_quality_flags',
}

def find_forbidden(obj: Any, path: str = '') -> list[str]:
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f'{path}.{k}' if path else k
            if k in FORBIDDEN:
                hits.append(p)
            hits.extend(find_forbidden(v, p))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(find_forbidden(v, f'{path}[{i}]'))
    return hits

qc = []
with output_jsonl.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue
        rec = json.loads(line)
        e = rec['llm_enrichment']
        qc.append({
            'citation': rec['citation'],
            'status': rec['llm_generation']['status'],
            'forbidden_fields': find_forbidden(rec),
            'concept_count': len(e.get('concepts_en', [])),
            'terms_original_count': len(e.get('terms_original', [])),
            'has_topic': bool(e.get('topic') or e.get('subtopic') or e.get('micro_topic')),
            'specificity_score': e.get('specificity_score'),
        })
qc_df = pd.DataFrame(qc)
display(qc_df.head(100))
print('Rows checked:', len(qc_df))
print('Forbidden field rows:', int(qc_df['forbidden_fields'].apply(bool).sum()))
print('Failed rows:', int(qc_df['status'].str.startswith('failed').sum()))
print('Status counts:', qc_df['status'].value_counts().to_dict())


,citation,status,forbidden_fields,concept_count,terms_original_count,has_topic,specificity_score
0,BGE 139 I 2 E. 5.7,ok,[],5,5,True,0.2
1,BGE 139 I 2 E. 6.3,ok,[],5,5,True,0.8
2,BGE 145 I 1 E. 1.1.3,ok,[],5,5,True,0.8
3,BGE 145 I 1 E. 6,ok,[],5,5,True,0.8
4,BGE 145 I 1 E. 6.5.1,ok,[],4,5,True,0.8
...,...,...,...,...,...,...,...
95,BGE 145 I 227 E. 6.6,ok,[],5,5,True,0.8
96,BGE 148 I 210 E. 4.4.5,ok,[],5,5,True,0.8
97,BGE 148 I 210 E. 4.5.3,ok,[],5,5,True,0.8
98,BGE 142 I 162 E. 3.2.1,ok,[],5,5,True,0.8


Rows checked: 363258
Forbidden field rows: 0
Failed rows: 1
Status counts: {'ok': 363212, 'ok_after_retry': 45, 'failed_descriptor_parse': 1}


## Tuning notes

If you see OOM during graph capture or first generation:
- drop `cfg.gpu_memory_utilization` to 0.88
- or drop `cfg.max_num_seqs` to 256

If FlashInfer didn't load and you ended up on Triton again:
- check the install log in Cell 0; the most common cause is a torch/CUDA version mismatch with the FlashInfer wheel
- you can also try `pip install flashinfer-python==0.2.x` pinning a known-good version
- TRITON_ATTN still benefits from all the other changes (FP8 KV, larger `max_num_seqs`, mega-batching, prefix caching, n-gram speculation) — expect ~3–4× over the old notebook even without FlashInfer

If you want even more throughput later, the next lever is `cfg.use_structured_outputs = True` (guided JSON via xgrammar). It eliminates parse failures entirely and constrains the decode at each step. It is currently OFF by default to keep this run a strict apples-to-apples comparison.


Python 3 Google Compute Engine backend (GPU)
Showing resources from 1:02 AM to 1:28 AM
System RAM
17.1 / 176.9 GB

GPU RAM
89.8 / 95.6 GB

Disk
72.2 / 235.7 GB
